In [ ]:
# import libraries
import os
import sys
sys.path.insert(0, '..')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif

from src.data.load_data import load_raw_data, encode_target
from src.data.clean_data import clean_pipeline
from src.features.build_features import build_features_pipeline

In [ ]:
# load raw data
data = encode_target(load_raw_data('../data/raw/bank-full.csv'))
cleaned = clean_pipeline(data)
print("Data loaded and cleaned successfully.")
print(f"Data shape: {cleaned.shape}")

In [ ]:
train_data, temp_data = train_test_split(cleaned, test_size=0.2, random_state=42, stratify=cleaned['y'])  # train-test split
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42, stratify=temp_data['y'])  # validation-test split
print(f"Training data shape: {train_data.shape}")
print(f"Validation data shape: {val_data.shape}")
print(f"Test data shape: {test_data.shape}")

train_features, train_target = build_features_pipeline(train_data, {'val' : val_data, 'test' : test_data})
new_columns = [c for c in train_features.columns if c not in train_data.columns]
print(f"New columns created: {new_columns}")
train_features[new_columns].head()

In [ ]:
# Confirm the seasonal prior lookup came from TRAIN only
month_rate_in_train = train_features.groupby('month')['Seasonal_Conversion_Prior'].first()
actual_train_target_rate_by_month = train_features.groupby('month')['y'].mean()
comparison_df = pd.DataFrame({'Feature_value': month_rate_in_train, 'Actual_Train_Rate': actual_train_target_rate_by_month})
print(comparison_df)

In [ ]:
# Mutual information: which features carry the most signal about the target
numeric_check_cols = ['age','balance','campaign', 'previous', 'days_since_contact',
                      'contact_recency_score', 'campaign_intensity',
                      'seasonal_conversion_prior', 'is_high_season']
mi_scores = mutual_info_classif(train_features[numeric_check_cols], train_features['y'], random_state=42)
pd.Series(mi_scores, index=numeric_check_cols).sort_values(ascending=False)

In [ ]:
# Cramer's V: association strength between categorical features and the target
from scipy.stats import chi2_contingency
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    min_dim = min(confusion_matrix.shape) - 1
    return (chi2 / n*min_dim) ** 0.5

for col in ['poutcome', 'job', 'contact', 'age_life_stage', 'contact_channel_trust']:
    v = cramers_v(train_features[col], train_features['y'])
    print(f"Cramer's V between {col} and target: {v:.4f}")